In [534]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [535]:
df = pd.read_csv('./data/course_lead_scoring.csv')

In [536]:
df

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1
...,...,...,...,...,...,...,...,...,...
1457,referral,manufacturing,1,NaN,self_employed,north_america,4,0.53,1
1458,referral,technology,3,65259.0,student,europe,2,0.24,1
1459,paid_ads,technology,1,45688.0,student,north_america,3,0.02,1
1460,referral,NaN,5,71016.0,self_employed,north_america,0,0.25,1


In [537]:
categorical_columns = list(df.columns[df.dtypes == 'object'])
numerical_columns = list(df.columns[(df.dtypes != 'object') & (df.columns != 'converted')])

for col in categorical_columns:
    df[col] = df[col].fillna('NA')

for col in numerical_columns:
    df[col] = df[col].fillna(0.0)

In [538]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [539]:
df_train

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score
0,events,manufacturing,2,95543.0,unemployed,europe,3,0.78
1,referral,NA,1,54924.0,student,south_america,6,0.39
2,organic_search,healthcare,2,77352.0,unemployed,europe,2,0.22
3,paid_ads,other,2,34600.0,employed,south_america,2,0.31
4,paid_ads,education,0,43615.0,unemployed,south_america,2,0.01
...,...,...,...,...,...,...,...,...
871,NA,other,5,67314.0,NA,europe,2,0.87
872,events,education,6,63996.0,NA,australia,4,0.92
873,organic_search,finance,1,73702.0,unemployed,north_america,2,0.55
874,events,technology,1,93341.0,student,middle_east,4,0.99


In [540]:
from sklearn.metrics import roc_auc_score

for feature in numerical_columns:
    auc = roc_auc_score(y_train, df_train[feature])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[feature])
    print(f"{feature}: AUC = {auc:.3f}")

number_of_courses_viewed: AUC = 0.764
annual_income: AUC = 0.552
interaction_count: AUC = 0.738
lead_score: AUC = 0.614


In [541]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='lbfgs', C=1.0, max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [542]:
y_pred = model.predict_proba(X_val)[:, 1]

In [543]:
roc_auc_score(y_val, y_pred)

0.9200460166810468

In [544]:
scores = []

thresholds = np.linspace(0, 1, 1001)


for t in thresholds:
    actual_positive = (y_val == 1)
    actual_negative = (y_val == 0)
    
    predict_positive = (y_pred >= t)
    predict_negative = (y_pred < t)

    tp = (predict_positive & actual_positive).sum()
    tn = (predict_negative & actual_negative).sum()

    fp = (predict_positive & actual_negative).sum()
    fn = (predict_negative & actual_positive).sum()
    
    scores.append((t, tp, fp, fn, tn))


columns = ['threshold', 'tp', 'fp', 'fn', 'tn']
df_scores = pd.DataFrame(scores, columns=columns)

df_scores['tpr'] = df_scores.tp / (df_scores.tp + df_scores.fn)
df_scores['fpr'] = df_scores.fp / (df_scores.fp + df_scores.tn)


In [545]:
df_scores

,threshold,tp,fp,fn,tn,tpr,fpr
0,0.000,171,122,0,0,1.000000,1.000000
1,0.001,171,122,0,0,1.000000,1.000000
2,0.002,171,122,0,0,1.000000,1.000000
3,0.003,171,120,0,2,1.000000,0.983607
4,0.004,171,120,0,2,1.000000,0.983607
...,...,...,...,...,...,...,...
996,0.996,26,0,145,122,0.152047,0.000000
997,0.997,22,0,149,122,0.128655,0.000000
998,0.998,14,0,157,122,0.081871,0.000000
999,0.999,11,0,160,122,0.064327,0.000000


In [546]:
t = df_scores[df_scores['tp'] == df_scores['fp']]
t

,threshold,tp,fp,fn,tn,tpr,fpr
1000,1.0,0,0,171,122,0.0,0.0


In [ ]:
from sklearn.metrics import f1_score

thresholds = np.arange(0.0, 1.01, 0.01)
f1_scores = []

for t in thresholds:
    y_pred = (y_pred >= t).astype(int)
    f1 = f1_score(y_val, y_pred)
    f1_scores.append(f1)

f1_scores = np.array(f1_scores)
f1_scores

array([0.73706897, 0.73706897, 0.73706897, 0.73706897, 0.73706897,
       0.73706897, 0.73706897, 0.73706897, 0.73706897, 0.73706897,
       0.73706897, 0.73706897, 0.73706897, 0.73706897, 0.73706897,
       0.73706897, 0.73706897, 0.73706897, 0.73706897, 0.73706897,
       0.73706897, 0.73706897, 0.73706897, 0.73706897, 0.73706897,
       0.73706897, 0.73706897, 0.73706897, 0.73706897, 0.73706897,
       0.73706897, 0.73706897, 0.73706897, 0.73706897, 0.73706897,
       0.73706897, 0.73706897, 0.73706897, 0.73706897, 0.73706897,
       0.73706897, 0.73706897, 0.73706897, 0.73706897, 0.73706897,
       0.73706897, 0.73706897, 0.73706897, 0.73706897, 0.73706897,
       0.73706897, 0.73706897, 0.73706897, 0.73706897, 0.73706897,
       0.73706897, 0.73706897, 0.73706897, 0.73706897, 0.73706897,
       0.73706897, 0.73706897, 0.73706897, 0.73706897, 0.73706897,
       0.73706897, 0.73706897, 0.73706897, 0.73706897, 0.73706897,
       0.73706897, 0.73706897, 0.73706897, 0.73706897, 0.73706

In [549]:
# หา threshold ที่ F1-max
best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]
best_f1

np.float64(0.7370689655172413)

In [551]:
from sklearn.model_selection import KFold

def train(df_train, y_train, C):
    dicts = df_train[categorical_columns + numerical_columns].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
    model.fit(X_train, y_train)
    
    return dv, model

def predict(df, dv, model):
    dicts = df[categorical_columns + numerical_columns].to_dict(orient='records')

    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred

In [552]:
n_splits = 5

for C in [0.000001, 0.001, 1, 5]:
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=1)

    scores = []

    for train_idx, val_idx in kfold.split(df_full_train):
        # the k-fold split uses index to shuffle the data
        df_train = df_full_train.iloc[train_idx]
        df_val = df_full_train.iloc[val_idx]
        
        # y values come from dataset
        y_train = df_train.converted.values
        y_val = df_val.converted.values
        
        # training and predicting
        dv, model = train(df_train, y_train, C=C)
        y_pred = predict(df_val, dv, model)
        
        # AUC
        auc = roc_auc_score(y_val, y_pred)
        scores.append(auc)

    print('C=%s %.3f +- %.3f' % (C, np.mean(scores), np.std(scores)))

C=1e-06 0.560 +- 0.024
C=0.001 0.867 +- 0.029
C=1 0.822 +- 0.036
C=5 0.822 +- 0.036
